<a href="https://colab.research.google.com/github/shema-boris/Text--SQL/blob/main/notebooks/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/shema-boris/Text--SQL.git


Cloning into 'Text--SQL'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 80 (delta 32), reused 69 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 13.16 MiB | 19.39 MiB/s, done.
Resolving deltas: 100% (32/32), done.


In [2]:
%cd Text--SQL


/content/Text--SQL


In [3]:
!ls
!pip install -r requirements.txt

checkpoints  notebooks	requirements.txt  src
data	     README.md	scripts		  tests.py


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
!pip install kagglehub

CUDA available: True


In [5]:
import kagglehub, shutil
from pathlib import Path

path = kagglehub.dataset_download("shahrukhkhan/wikisql")
print("Downloaded to:", path)

src = Path(path)
dst = Path("/content/data/external/wikisql")
dst.mkdir(parents=True, exist_ok=True)

for p in src.iterdir():
    shutil.copy(p, dst / p.name)

!ls /content/data/external/wikisql

100%|██████████| 7.12M/7.12M [00:00<00:00, 228MB/s]

Extracting files...
Downloaded to: /root/.cache/kagglehub/datasets/shahrukhkhan/wikisql/versions/2


test.csv   validation.csv     wikisql_train.json
train.csv  wikisql_test.json  wikisql_validation.json


In [6]:
!python scripts/prepare_wikisql.py
!ls data/raw

dev.jsonl    train.jsonl	wikisql_test.jsonl
testt.jsonl  wikisql_dev.jsonl	wikisql_train.jsonl


In [ ]:
from pathlib import Path

src = Path("data/raw/wikisql_train.jsonl")
dst = Path("data/raw/wikisql_train_small.jsonl")

max_lines = 25000  # or 1000 for very fast tests

with src.open("r", encoding="utf-8") as f_in, dst.open("w", encoding="utf-8") as f_out:
    for i, line in enumerate(f_in):
        if i >= max_lines:
            break
        f_out.write(line)

print("Wrote", max_lines, "lines to", dst)

Wrote 25000 lines to data/raw/wikisql_train_small.jsonl


In [10]:
!python -m src.training.trainer

Epoch 1: train_loss = 6.4733, dev_loss = 5.8364
Epoch 2: train_loss = 4.2532, dev_loss = 5.2662
Epoch 3: train_loss = 3.0523, dev_loss = 5.4332
Epoch 4: train_loss = 2.3357, dev_loss = 5.2357
Epoch 5: train_loss = 1.8710, dev_loss = 5.3720
Epoch 6: train_loss = 1.6762, dev_loss = 5.2807
Epoch 7: train_loss = 1.5107, dev_loss = 5.3385
Epoch 8: train_loss = 1.3828, dev_loss = 5.5732
Epoch 9: train_loss = 1.2864, dev_loss = 5.6361
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/Text--SQL/src/training/trainer.py", line 261, in <module>
    main()
  File "/content/Text--SQL/src/training/trainer.py", line 236, in main
    train_loss = train_one_epoch(
                 ^^^^^^^^^^^^^^^^
  File "/content/Text--SQL/src/training/trainer.py", line 129, in train_one_epoch
    loss.backward()
  File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 625, in backward
    torch.au

In [ ]:
!python -m src.training.inference

Enter a question (or 'quit' to exit):
> How many schools did player number 3 play at?
Predicted SQL:
select count from from table where = 3 3
> What position does the player who played for butler cc (ks) play?
Predicted SQL:
select position from table where position = = 16
> 

In [9]:
!head -n 3 data/raw/wikisql_dev.jsonl

{"question": "What position does the player who played for butler cc (ks) play?", "sql": "SELECT Position FROM table WHERE School/Club Team = Butler CC (KS)"}
{"question": "How many schools did player number 3 play at?", "sql": "SELECT COUNT School/Club Team FROM table WHERE No. = 3"}
{"question": "What school did player number 21 play for?", "sql": "SELECT School/Club Team FROM table WHERE No. = 21"}


In [11]:
import json
import torch

from src.data.tokenizer import Vocabulary
from src.data.dataset import TextToSQLDataset
from src.models.encoder import Encoder
from src.models.decoder import Decoder
from src.models.seq2seq import Seq2Seq
from src.training.inference import build_vocabs_from_dataset, build_model, translate_question

PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths – change if yours are different
train_path = "data/raw/wikisql_train.jsonl"  # training JSONL you used for training
dev_path   = "data/raw/wikisql_dev.jsonl"    # dev JSONL
checkpoint_path = "checkpoints/text2sql.pt"

max_src_len = 64
max_trg_len = 64

# 1) Rebuild vocabs from training data (same as in inference.py)
src_vocab, trg_vocab = build_vocabs_from_dataset(train_path, max_src_len, max_trg_len)

# 2) Build model and load weights
model = build_model(src_vocab, trg_vocab, device)
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

RuntimeError: Error(s) in loading state_dict for Seq2Seq:
	size mismatch for encoder.embedding.weight: copying a param with shape torch.Size([33, 128]) from checkpoint, the shape in current model is torch.Size([53241, 128]).
	size mismatch for decoder.embedding.weight: copying a param with shape torch.Size([26, 128]) from checkpoint, the shape in current model is torch.Size([41375, 128]).
	size mismatch for decoder.fc_out.weight: copying a param with shape torch.Size([26, 768]) from checkpoint, the shape in current model is torch.Size([41375, 768]).
	size mismatch for decoder.fc_out.bias: copying a param with shape torch.Size([26]) from checkpoint, the shape in current model is torch.Size([41375]).

In [ ]:
N = 10  # how many dev examples to inspect
count = 0

with open(dev_path, "r", encoding="utf-8") as f:
    for line in f:
        if count >= N:
            break

        ex = json.loads(line)
        question = ex["question"]
        gold_sql = ex["sql"]

        pred_sql = translate_question(
            model=model,
            src_vocab=src_vocab,
            trg_vocab=trg_vocab,
            question=question,
            device=device,
        )

        print("=" * 80)
        print(f"Example {count+1}")
        print(f"QUESTION : {question}")
        print(f"GOLD SQL : {gold_sql}")
        print(f"PRED SQL : {pred_sql}")
        count += 1

Example 1
QUESTION : What position does the player who played for butler cc (ks) play?
GOLD SQL : SELECT Position FROM table WHERE School/Club Team = Butler CC (KS)
PRED SQL : select position from table where player for = and player = =
Example 2
QUESTION : How many schools did player number 3 play at?
GOLD SQL : SELECT COUNT School/Club Team FROM table WHERE No. = 3
PRED SQL : select count team from table where = 3
Example 3
QUESTION : What school did player number 21 play for?
GOLD SQL : SELECT School/Club Team FROM table WHERE No. = 21
PRED SQL : select school team table where where = =
Example 4
QUESTION : Who is the player that wears number 42?
GOLD SQL : SELECT Player FROM table WHERE No. = 42
PRED SQL : select player from table where points = =
Example 5
QUESTION : What player played guard for toronto in 1996-97?
GOLD SQL : SELECT Player FROM table WHERE Position = Guard AND Years in Toronto = 1996-97
PRED SQL : select player from table where position = guard and school/club = =

In [ ]:
import json
from typing import List, Tuple

import torch
import torch.nn as nn

from src.data.tokenizer import Vocabulary
from src.data.dataset import TextToSQLDataset
from src.models.encoder import Encoder
from src.models.decoder import Decoder
from src.models.seq2seq import Seq2Seq
from src.training.inference import (
    build_vocabs_from_dataset,
    build_model,
    translate_question,
)

# ----------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------

PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Adjust these paths if yours are different
train_path = "data/raw/wikisql_train.jsonl"   # training JSONL (same one you trained on)
dev_path = "data/raw/wikisql_dev.jsonl"       # dev JSONL
checkpoint_path = "checkpoints/text2sql.pt"   # trained model weights

max_src_len = 64
max_trg_len = 64

# ----------------------------------------------------------------------
# Metrics
# ----------------------------------------------------------------------


def sql_exact_match(gold: str, pred: str) -> float:
    """
    Exact string match after simple strip.
    Returns 1.0 if equal, else 0.0
    """
    return 1.0 if gold.strip() == pred.strip() else 0.0


def sql_token_accuracy(gold: str, pred: str) -> float:
    """
    Very simple token-level accuracy:
    - Tokenize both gold and pred by whitespace
    - Compare tokens position-wise up to len(gold_tokens)
    - Accuracy = (# of matching positions) / max(1, len(gold_tokens))

    This is not a perfect SQL metric, but it gives a rough idea.
    """
    gold_tokens = gold.strip().split()
    pred_tokens = pred.strip().split()

    if not gold_tokens:
        return 1.0 if not pred_tokens else 0.0

    matches = 0
    for i, g in enumerate(gold_tokens):
        if i < len(pred_tokens) and pred_tokens[i] == g:
            matches += 1

    return matches / max(1, len(gold_tokens))


# ----------------------------------------------------------------------
# Simple dev evaluation: examples + aggregated metrics
# ----------------------------------------------------------------------


def evaluate_dev(
    model: Seq2Seq,
    src_vocab: Vocabulary,
    trg_vocab: Vocabulary,
    dev_path: str,
    num_examples_to_print: int = 10,
    max_examples_for_metrics: int = 1000,
) -> Tuple[float, float]:
    """
    Iterate over dev JSONL, run the model, and compute:
      - exact match rate
      - token-level accuracy

    Also prints the first `num_examples_to_print` triplets for inspection.
    """

    total_em = 0.0
    total_tok_acc = 0.0
    n = 0

    print_count = 0

    with open(dev_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            ex = json.loads(line)
            question = ex["question"]
            gold_sql = ex["sql"]

            # Run model to get predicted SQL
            pred_sql = translate_question(
                model=model,
                src_vocab=src_vocab,
                trg_vocab=trg_vocab,
                question=question,
                device=device,
            )

            # Compute metrics
            em = sql_exact_match(gold_sql, pred_sql)
            tok_acc = sql_token_accuracy(gold_sql, pred_sql)

            total_em += em
            total_tok_acc += tok_acc
            n += 1

            # Print a few examples for qualitative analysis
            if print_count < num_examples_to_print:
                print("=" * 80)
                print(f"Example {print_count + 1}")
                print(f"QUESTION : {question}")
                print(f"GOLD SQL : {gold_sql}")
                print(f"PRED SQL : {pred_sql}")
                print(f"EXACT MATCH : {em:.1f}")
                print(f"TOKEN ACC  : {tok_acc:.3f}")
                print_count += 1

            if n >= max_examples_for_metrics:
                break

    if n == 0:
        print("No examples found in dev file!")
        return 0.0, 0.0

    em_rate = total_em / n
    avg_tok_acc = total_tok_acc / n

    print("-" * 80)
    print(f"Evaluated on {n} dev examples")
    print(f"Exact Match Rate     : {em_rate:.4f}")
    print(f"Token-level Accuracy : {avg_tok_acc:.4f}")

    return em_rate, avg_tok_acc


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------


def main():
    print(f"Using device: {device}")

    # 1) Rebuild vocabs from training data (same logic as inference.py)
    print("Building vocabularies from training data...")
    src_vocab, trg_vocab = build_vocabs_from_dataset(
        path=train_path,
        max_src_len=max_src_len,
        max_trg_len=max_trg_len,
    )

    # 2) Build model and load trained weights
    print("Building model...")
    model = build_model(src_vocab, trg_vocab, device)

    print(f"Loading checkpoint from {checkpoint_path}...")
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    # 3) Evaluate on dev set
    print("Evaluating on dev set...")
    evaluate_dev(
        model=model,
        src_vocab=src_vocab,
        trg_vocab=trg_vocab,
        dev_path=dev_path,
        num_examples_to_print=10,
        max_examples_for_metrics=1000,
    )


if __name__ == "__main__":
    main()

Using device: cuda
Building vocabularies from training data...
Building model...
Loading checkpoint from checkpoints/text2sql.pt...
Evaluating on dev set...
Example 1
QUESTION : What position does the player who played for butler cc (ks) play?
GOLD SQL : SELECT Position FROM table WHERE School/Club Team = Butler CC (KS)
PRED SQL : select position from table where position = = 16
EXACT MATCH : 0.0
TOKEN ACC  : 0.182
Example 2
QUESTION : How many schools did player number 3 play at?
GOLD SQL : SELECT COUNT School/Club Team FROM table WHERE No. = 3
PRED SQL : select count from from table where = 3 3
EXACT MATCH : 0.0
TOKEN ACC  : 0.000
Example 3
QUESTION : What school did player number 21 play for?
GOLD SQL : SELECT School/Club Team FROM table WHERE No. = 21
PRED SQL : select school from table where player = 21 21
EXACT MATCH : 0.0
TOKEN ACC  : 0.111
Example 4
QUESTION : Who is the player that wears number 42?
GOLD SQL : SELECT Player FROM table WHERE No. = 42
PRED SQL : select player fro

In [ ]:
!ls



checkpoints  data  notebooks  requirements.txt	scripts  src  tests.py
